In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys

# sys.path.insert(0, "/home/matis/code/Florian-Q/maraicherbio-prediction/notebooks/")

In [3]:
import utils
import utils_series
import pandas as pd
import numpy as np
import itertools
import matplotlib.pyplot as plt
import model_prophet


/home/flo/.pyenv/versions/maraicherbio-prediction/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Importing plotly failed. Interactive plots will not work.


In [4]:
# --- Boucle : df_train / df_test pour chaque produit (split adaptatif, date fin commune) ---
df_all = utils.charger_dataframe()

# Définir la date de fin commune pour TOUS les splits
utils_series.GLOBAL_TEST_END_DATE = df_all['created'].max()
print(f'Date de fin commune : {utils_series.GLOBAL_TEST_END_DATE.date()}\n')

modeles = sorted(df_all['model'].unique())
print(f'{len(modeles)} produits à traiter\n')

dict_train = {}
dict_test = {}
skipped = []

for modele in modeles:
    try:
        df_produit = utils.charger_dataframe(modele)
        df_model = utils_series.complete_weekly_dataframe(df_produit, 'created', 'quantite_y')
        train, test = utils_series.split_adaptive_seasonal(df_model, test_pct=0.20)
        dict_train[modele] = train
        dict_test[modele] = test
    except ValueError as e:
        skipped.append(modele)
        continue

print(f'\nTerminé : {len(dict_train)} produits prêts  |  {len(skipped)} ignorés')
if skipped:
    print(f'Ignorés : {skipped}')

# Vérification
test_ends = [t.index.max().date() for t in dict_test.values()]
print(f'Toutes les fins de test = {test_ends[0]} ?  {len(set(test_ends)) == 1}')

Date de fin commune : 2026-05-27

100 produits à traiter

Train : 2014-06-08  →  2024-05-26  (521 semaines)
Test  : 2024-06-02   →  2026-05-24   (104 semaines)
  → test=2 an(s) sur 12.0 ans (17%)
Train : 2014-07-06  →  2024-05-26  (517 semaines)
Test  : 2024-06-02   →  2026-05-24   (104 semaines)
  → test=2 an(s) sur 11.9 ans (17%)
Train : 2014-06-29  →  2024-05-26  (518 semaines)
Test  : 2024-06-02   →  2026-05-31   (105 semaines)
  → test=2 an(s) sur 11.9 ans (17%)
Train : 2015-01-11  →  2024-05-26  (490 semaines)
Test  : 2024-06-02   →  2026-05-24   (104 semaines)
  → test=2 an(s) sur 11.4 ans (18%)
Train : 2018-05-06  →  2024-05-26  (317 semaines)
Test  : 2024-06-02   →  2026-05-31   (105 semaines)
  → test=2 an(s) sur 8.1 ans (25%)
Train : 2014-01-05  →  2024-05-26  (543 semaines)
Test  : 2024-06-02   →  2026-05-31   (105 semaines)
  → test=2 an(s) sur 12.4 ans (16%)
Train : 2014-02-23  →  2024-05-26  (536 semaines)
Test  : 2024-06-02   →  2026-05-24   (104 semaines)
  → test=2 an

In [8]:
# ============================================================
# PROPHET v1 : Moyenne hebdo + split_adaptive + seasonal_metrics
# ============================================================

# ── Hyperparamètres fixes (validés par expérimentation) ──────────────────────
BEST_PARAMS: dict = {
    "growth"                  : "flat",
    "seasonality_mode"        : "additive",
    "weekly_seasonality"      : True,
    "daily_seasonality"       : False,
    "yearly_seasonality"      : True,
    "interval_width"          : 0.95,
    "changepoint_prior_scale" : 0.08,   # fixé — meilleur sur tous les produits
    "holidays_prior_scale"    : 1,      # fixé — meilleur sur tous les produits
}

# ── Grid Search (1 seul paramètre restant) ────────────────────────────────────
TUNING_GRID: dict = {
    "seasonality_prior_scale" : [3, 10],   # varie selon les produits
}

ALL_COMBOS = list(itertools.product(
    TUNING_GRID["seasonality_prior_scale"],
))
# 2 combinaisons par produit



# ── MODE ÉVALUATION (remplace la boucle Prophet v2) ──────────────────────────
results = []

for modele in dict_train.keys():
    y_train = dict_train[modele]['quantite_y']
    y_test  = dict_test[modele]['quantite_y']

    best_mape = np.inf
    best_sps     = None

    for (sps,) in ALL_COMBOS:
        try:
            metrics = model_prophet.run_prophet(sps, y_train, y_test)
            if metrics["MAPE"] < best_mape:
                best_mape   = metrics["MAPE"]
                best_sps     = sps
        except Exception as e:
            print(f"[WARN] {modele} | sps={sps} → {e}")
            continue

    if best_mape is None:
        continue

    results.append({
        'produit'    : modele,
        'train_debut': y_train.index.min().strftime('%Y-%m'),
        'best_sps'   : best_sps,
        'MAPE'       : round(best_mape,      1),
    })

df_prophet = pd.DataFrame(results)

# ── MODE PRÉDICTION (exemple sur un produit) ─────────────────────────────────
# df_pred = run_prophet(10, dict_train["mon_produit"])
# print(df_pred)

09:42:40 - cmdstanpy - INFO - Chain [1] start processing
09:42:40 - cmdstanpy - INFO - Chain [1] done processing
09:42:40 - cmdstanpy - INFO - Chain [1] start processing
09:42:40 - cmdstanpy - INFO - Chain [1] done processing
09:42:40 - cmdstanpy - INFO - Chain [1] start processing
09:42:40 - cmdstanpy - INFO - Chain [1] done processing
09:42:41 - cmdstanpy - INFO - Chain [1] start processing
09:42:41 - cmdstanpy - INFO - Chain [1] done processing
09:42:41 - cmdstanpy - INFO - Chain [1] start processing
09:42:41 - cmdstanpy - INFO - Chain [1] done processing
09:42:41 - cmdstanpy - INFO - Chain [1] start processing
09:42:41 - cmdstanpy - INFO - Chain [1] done processing
09:42:41 - cmdstanpy - INFO - Chain [1] start processing
09:42:41 - cmdstanpy - INFO - Chain [1] done processing
09:42:41 - cmdstanpy - INFO - Chain [1] start processing
09:42:41 - cmdstanpy - INFO - Chain [1] done processing
09:42:41 - cmdstanpy - INFO - Chain [1] start processing
09:42:41 - cmdstanpy - INFO - Chain [1]

In [11]:
df_prophet['MAPE'].mean()

np.float64(74.18809523809524)

In [7]:
df_prophet.to_csv('../data/Prophet_metrics.csv', index=False)